In [ ]:
import os
from dotenv import load_dotenv
from pageindex import PageIndexClient
import pageindex.utils as utils

load_dotenv()

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.environ["PAGEINDEX_API_KEY"]
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [ ]:
import openai

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

async def call_llm(prompt, model="gpt-4.1", temperature=0):
    client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

In [4]:
# ...existing code...
import os

pdf_path = r"sample_doc_demo.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print("Document Submitted:", doc_id)
# ...existing code...

Document Submitted: pi-cmmgo38hz03bh6eqnirggme2m


In [5]:
# ...existing code...
import time

max_wait_seconds = 300
poll_interval = 5
start = time.time()

while time.time() - start < max_wait_seconds:
    if pi_client.is_retrieval_ready(doc_id):
        tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
        print("Simplified Tree Structure of the Document:")
        utils.print_tree(tree)
        break

    print("Still processing...")
    time.sleep(poll_interval)
else:
    print("Timed out waiting for PageIndex processing.")
# ...existing code...

Still processing...
Still processing...
Simplified Tree Structure of the Document:
[{'title': 'Llama 2: Open Foundation and Fine-Tuned ...',
  'node_id': '0000',
  'prefix_summary': 'The text is a title page for a document ...',
  'nodes': [{'title': 'Abstract',
             'node_id': '0001',
             'summary': 'This text announces the release of Llama...'}]},
 {'title': 'Contents', 'node_id': '0002', 'summary': 'The document outlines the structure of a...'},
 {'title': '1 Introduction',
  'node_id': '0003',
  'summary': 'This text introduces Large Language Mode...'},
 {'title': '2 Pretraining',
  'node_id': '0004',
  'summary': '# 2 Pretraining\n\nTo create the new famil...'},
 {'title': '2.1 Pretraining Data',
  'node_id': '0005',
  'summary': '# 2.1 Pretraining Data\n\nOur training cor...'},
 {'title': '2.2 Training Details',
  'node_id': '0006',
  'summary': 'The text details the training setup for ...'},
 {'title': '2.2.1 Training Hardware &amp; Carbon Foo...',
  'node_id': 

In [6]:
import json

query = "What is the solution in paper about the tradeoff between helpfulness and safety?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt)

In [7]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks about the solution in the paper regarding the tradeoff between helpfulness and
safety. The summaries indicate that Section 3.2.2 'Reward Modeling' specifically addresses the
tradeoff by mentioning that two separate reward models are trained: one for helpfulness and one for
safety. This is a direct solution to the tradeoff. Section 3.2 'Reinforcement Learning with Human
Feedback (RLHF)' provides context on how human feedback is used to align model behavior, which is
also relevant. Section 3.2.1 discusses data collection for both helpfulness and safety, which is
part of the solution process. The Abstract and Introduction mention the tradeoff at a high level but
do not detail the solution. Therefore, the most relevant nodes are 0013 (Reward Modeling), 0011
(RLHF), and 0012 (Human Preference Data Collection).

Retrieved Nodes:
Node ID: 0011	 Page: 9	 Title: 3.2 Reinforcement Learning with Human Feedback (RLHF)
Node ID: 0012	 Page: 10	 Title: 3.2.1 Human

In [8]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

# 3.2 Reinforcement Learning with Human Feedback (RLHF)

RLHF is a model training procedure that is applied to a fine-tuned language model to further align
model behavior with human preferences and instruction following. We collect data that represents
empirically

sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the
preferences of the human annotators and can then automate preference decisions.


#### 3.2.1 Human Preference Data Collection

Next, we collect human preference data for reward modeling. We chose a binary comparison protocol
over other schemes, mainly because it enables us to maximize the diversity of collected prompts.
Still, other strategies are worth considering, which we leave for future work.

Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between two sampl

In [9]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await call_llm(answer_prompt)
utils.print_wrapped(answer)

Generated Answer:

**Solution to the tradeoff between helpfulness and safety:**

The paper addresses the tradeoff between helpfulness and safety by **training two separate reward
models**: one optimized for helpfulness (Helpfulness RM) and another for safety (Safety RM). This
approach is taken because helpfulness and safety can sometimes conflict, making it difficult for a
single reward model to perform well on both objectives. By separating the reward models, each can be
specifically optimized and guided by distinct annotation criteria and data, allowing for better
alignment with human preferences in both helpfulness and safety.
